# SMS Spam Detection System

A clean, interview-friendly notebook for the final project.

**Goal:** classify SMS messages as **Ham (0)** or **Spam (1)** and build the same optimized model artifact used by the Streamlit application.

### Final modeling flow
Data Cleaning → EDA → Baseline MultinomialNB → Character TF-IDF + LinearSVC → Cross-Validation → GridSearchCV → Threshold Optimization → Error Analysis → Probability Calibration → Save Final Artifact


## 1. Imports

The final classifier works directly on raw SMS text.  
We do **not** manually stem or remove stopwords before inference, which keeps training and app preprocessing consistent.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from wordcloud import WordCloud

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV,
    cross_val_predict,
)
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    average_precision_score,
    precision_recall_curve,
    brier_score_loss,
)


## 2. Load and Clean the Dataset


In [ ]:
df = pd.read_csv("spam.csv", encoding="latin1")

# Keep only the useful columns
df = df[["v1", "v2"]].rename(
    columns={
        "v1": "target",
        "v2": "text",
    }
)

# Encode labels: ham = 0, spam = 1
df["target"] = df["target"].map(
    {
        "ham": 0,
        "spam": 1,
    }
)

# Remove missing values and duplicate messages
df = (
    df.dropna(subset=["text", "target"])
      .drop_duplicates(keep="first")
      .reset_index(drop=True)
)

print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df["target"].value_counts())


After duplicate removal, the original project run contained **5,169 messages**:
- **4,516 Ham**
- **653 Spam**

The dataset is imbalanced, so accuracy alone is not enough. We also track **precision, recall, F1-score, confusion matrix, and PR-AUC**.


## 3. Exploratory Data Analysis


In [ ]:
class_counts = df["target"].value_counts().sort_index()

plt.figure(figsize=(5, 4))
plt.bar(["Ham", "Spam"], [class_counts[0], class_counts[1]])
plt.title("Class Distribution")
plt.ylabel("Number of Messages")
plt.show()


In [ ]:
df["message_length"] = df["text"].str.len()

plt.figure(figsize=(8, 4))
plt.hist(
    [
        df.loc[df["target"] == 0, "message_length"],
        df.loc[df["target"] == 1, "message_length"],
    ],
    bins=40,
    label=["Ham", "Spam"],
    alpha=0.7,
)
plt.xlabel("Message Length")
plt.ylabel("Frequency")
plt.title("Message Length Distribution")
plt.legend()
plt.show()


### WordClouds

These WordClouds are also saved as `ham_wc.png` and `spam_wc.png` for the Streamlit **Dataset Insights** page.


In [ ]:
ham_text = " ".join(
    df.loc[df["target"] == 0, "text"].astype(str).str.lower()
)

spam_text = " ".join(
    df.loc[df["target"] == 1, "text"].astype(str).str.lower()
)

ham_wc = WordCloud(
    width=800,
    height=400,
    background_color="white",
).generate(ham_text)

spam_wc = WordCloud(
    width=800,
    height=400,
    background_color="white",
).generate(spam_text)

ham_wc.to_file("ham_wc.png")
spam_wc.to_file("spam_wc.png")

plt.figure(figsize=(12, 5))
plt.imshow(ham_wc, interpolation="bilinear")
plt.axis("off")
plt.title("Ham WordCloud")
plt.show()

plt.figure(figsize=(12, 5))
plt.imshow(spam_wc, interpolation="bilinear")
plt.axis("off")
plt.title("Spam WordCloud")
plt.show()


## 4. Train/Test Split


In [ ]:
X = df["text"]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training examples:", len(X_train))
print("Test examples:", len(X_test))


## 5. Baseline Model — TF-IDF + Multinomial Naive Bayes

The baseline gives very high precision but misses too many spam messages.  
This provides a clear reason to improve **spam recall**.


In [ ]:
baseline_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=6708,
        ),
    ),
    (
        "model",
        MultinomialNB(),
    ),
])

baseline_pipeline.fit(X_train, y_train)
y_pred_baseline = baseline_pipeline.predict(X_test)

baseline_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_baseline),
    "Precision": precision_score(y_test, y_pred_baseline),
    "Recall": recall_score(y_test, y_pred_baseline),
    "F1": f1_score(y_test, y_pred_baseline),
}

for name, value in baseline_metrics.items():
    print(f"{name:9s}: {value:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_baseline))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_baseline))


## 6. Improved Model — Character TF-IDF + Linear SVM

Character n-grams are useful for noisy SMS patterns such as:

`FREE`, `FR33`, `W1NNER`, short codes, numbers, spelling variations, and obfuscated promotional text.


In [ ]:
char_svm_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="char",
            ngram_range=(3, 5),
            min_df=2,
            max_features=50000,
            sublinear_tf=True,
        ),
    ),
    (
        "model",
        LinearSVC(),
    ),
])

char_svm_pipeline.fit(X_train, y_train)
y_pred_char_svm = char_svm_pipeline.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred_char_svm))
print("Precision:", precision_score(y_test, y_pred_char_svm))
print("Recall   :", recall_score(y_test, y_pred_char_svm))
print("F1-score :", f1_score(y_test, y_pred_char_svm))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_char_svm))


## 7. 5-Fold Stratified Cross-Validation


In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "pr_auc": "average_precision",
}

cv_results = cross_validate(
    char_svm_pipeline,
    X,
    y,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
)

for metric in scoring:
    scores = cv_results[f"test_{metric}"]
    print(
        f"{metric:10s}: "
        f"{scores.mean():.4f} ± {scores.std():.4f}"
    )


## 8. Hyperparameter Tuning

GridSearchCV is run **only on the training split**.  
The optimization metric is **F1-score**, balancing spam precision and recall.


In [ ]:
svm_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="char",
            sublinear_tf=True,
        ),
    ),
    (
        "model",
        LinearSVC(),
    ),
])

param_grid = {
    "tfidf__ngram_range": [
        (3, 5),
        (3, 6),
        (2, 5),
    ],
    "tfidf__min_df": [1, 2],
    "model__C": [0.5, 1, 2],
}

grid_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest CV F1-score:")
print(grid_search.best_score_)


## 9. Evaluate the Tuned Model


In [ ]:
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred_best))
print("Precision:", precision_score(y_test, y_pred_best))
print("Recall   :", recall_score(y_test, y_pred_best))
print("F1-score :", f1_score(y_test, y_pred_best))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))

decision_scores = best_model.decision_function(X_test)
pr_auc = average_precision_score(
    y_test,
    decision_scores,
)

print("\nPR-AUC:", pr_auc)


## 10. Error Analysis


In [ ]:
results = pd.DataFrame({
    "text": X_test.values,
    "actual": y_test.values,
    "predicted": y_pred_best,
})

false_negatives = results[
    (results["actual"] == 1)
    & (results["predicted"] == 0)
]

false_positives = results[
    (results["actual"] == 0)
    & (results["predicted"] == 1)
]

print("Missed spam messages:", len(false_negatives))
print("False positives:", len(false_positives))

false_negatives[["text", "actual", "predicted"]]


The tuned SVM was still conservative: it had very few false positives but missed some legitimate-looking or short spam messages.  
Instead of changing the model again, we optimize the **decision threshold** using out-of-fold training scores.


## 11. Decision Threshold Optimization


In [ ]:
# Out-of-fold decision scores from TRAINING DATA only
oof_scores = cross_val_predict(
    clone(best_model),
    X_train,
    y_train,
    cv=cv,
    method="decision_function",
    n_jobs=-1,
)

precision_values, recall_values, thresholds = precision_recall_curve(
    y_train,
    oof_scores,
)

f1_values = (
    2 * precision_values[:-1] * recall_values[:-1]
    / (
        precision_values[:-1]
        + recall_values[:-1]
        + 1e-10
    )
)

# Keep precision at or above 98%
valid = precision_values[:-1] >= 0.98

best_index = np.argmax(
    np.where(valid, f1_values, -1)
)

best_threshold = thresholds[best_index]

print("Best threshold :", best_threshold)
print("CV Precision   :", precision_values[best_index])
print("CV Recall      :", recall_values[best_index])
print("CV F1-score    :", f1_values[best_index])


In [ ]:
test_scores = best_model.decision_function(X_test)

y_pred_threshold = (
    test_scores >= best_threshold
).astype(int)

final_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_threshold),
    "Precision": precision_score(y_test, y_pred_threshold),
    "Recall": recall_score(y_test, y_pred_threshold),
    "F1": f1_score(y_test, y_pred_threshold),
}

for name, value in final_metrics.items():
    print(f"{name:9s}: {value:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_threshold))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_threshold))


## 12. Baseline vs Final Model


In [ ]:
comparison = pd.DataFrame(
    {
        "Baseline": baseline_metrics,
        "Optimized": final_metrics,
    }
)

comparison


### Reference results from the original project run

| Metric | Baseline | Optimized |
|---|---:|---:|
| Accuracy | 95.45% | 99.32% |
| Precision | 100.00% | 98.44% |
| Recall | 64.12% | 96.18% |
| F1-score | 78.14% | 97.30% |
| Missed Spam | 47 | 5 |

Small numerical differences can occur after re-running in a different Python/scikit-learn environment.


## 13. Probability Calibration

`LinearSVC` produces decision scores, not probabilities.  
A separate calibrated classifier is trained so the Streamlit app can display a **Spam Risk Score**.


In [ ]:
calibrated_model = CalibratedClassifierCV(
    estimator=best_model,
    method="sigmoid",
    cv=5,
)

calibrated_model.fit(X_train, y_train)

spam_probability = calibrated_model.predict_proba(
    X_test
)[:, 1]

print("First 10 spam probabilities:")
print(spam_probability[:10])


In [ ]:
brier = brier_score_loss(
    y_test,
    spam_probability,
)

print("Brier Score:", brier)


## 14. Save the Final Artifact

The Streamlit application loads **one file** containing:

- tuned text-classification pipeline
- optimized decision threshold
- calibrated probability model

This avoids the old setup of saving a model and vectorizer separately.


In [ ]:
final_artifact = {
    "pipeline": best_model,
    "threshold": float(best_threshold),
    "calibrated_model": calibrated_model,
}

joblib.dump(
    final_artifact,
    "spam_classifier_optimized.pkl",
)

print("Final optimized model saved successfully.")


In [ ]:
artifact = joblib.load(
    "spam_classifier_optimized.pkl"
)

print("Artifact keys:", artifact.keys())
print("Threshold:", artifact["threshold"])


## 15. Final Project Flow

```text
Raw SMS
   ↓
Character-level TF-IDF
   ↓
Tuned Linear SVM
   ↓
Optimized Decision Threshold
   ↓
Spam / Ham Prediction
```

For the Streamlit app, the calibrated model additionally provides the **Spam Risk Score**.

The final application adds:
- sender block / unblock
- persistent SQLite blocklist
- prediction history
- Correct / Incorrect feedback

The app performs **SMS spam filtering**, not toxicity detection or real device-level SMS blocking.
